## Load Data

In [ ]:
import torch
from torch import nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.datasets import FashionMNIST
import torchvision.transforms as transforms
import numpy as np
import random

np.random.seed(0)
random.seed(0)
torch.manual_seed(0)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [ ]:
batch_size = 512
num_epochs = 15

train_dataset = FashionMNIST('./data', train=True, download=True, transform=transforms.ToTensor())
train_loader = DataLoader(train_dataset, batch_size, shuffle=True)

In [ ]:
# prompt: create SwiGLU

import torch
from torch import nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.datasets import FashionMNIST
import torchvision.transforms as transforms
import numpy as np
import random

# ## Load Data
class SwiGLU(nn.Module):
  def __init__(self):
    super(SwiGLU, self).__init__()
    self.sigmoid = nn.Sigmoid()

  def forward(self, x):
    a, b = x.chunk(2, dim=1)
    return a * self.sigmoid(b)

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dims, hidden_dims, output_dims):
        super(MLP, self).__init__()
        self.layer1 = nn.Linear(input_dims, hidden_dims *2)
        self.bn1 = nn.BatchNorm1d(hidden_dims *2)
        self.layer2 = nn.Linear(hidden_dims, hidden_dims *2)
        self.bn2 = nn.BatchNorm1d(hidden_dims *2)
        self.layer3 = nn.Linear(hidden_dims, hidden_dims *2)
        self.bn3 = nn.BatchNorm1d(hidden_dims *2)
        self.output = nn.Linear(hidden_dims, output_dims)

        for m in self.modules():
            if isinstance(m, nn.Linear):
                # nn.init.normal_(m.weight, mean=0.0, std=0.05)
                nn.init.kaiming_normal_(m.weight)

                nn.init.constant_(m.bias, 0.0)


    def forward(self, x):
        x = nn.Flatten()(x)
        x = self.layer1(x)
        x = self.bn1(x)
        x = SwiGLU()(x)
        identity = x
        x = self.layer2(x)
        x = self.bn2(x)
        x = SwiGLU()(x)
        x = self.layer3(x)
        x = self.bn3(x)
        x = SwiGLU()(x)
        x = x + identity
        out = self.output(x)

        return out

In [ ]:
print(model)

MLP(
  (layer1): Linear(in_features=784, out_features=128, bias=True)
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layer2): Linear(in_features=128, out_features=128, bias=True)
  (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layer3): Linear(in_features=128, out_features=128, bias=True)
  (bn3): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (output): Linear(in_features=128, out_features=10, bias=True)
)


In [ ]:
model = MLP(input_dims=784, hidden_dims=128, output_dims=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [ ]:

for epoch in range(num_epochs):
    t_loss = 0
    t_acc = 0
    cnt = 0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        t_loss += loss.item()
        t_acc += (torch.argmax(outputs, 1) == y).sum().item()
        cnt += len(y)

    t_loss /= len(train_loader)
    t_acc /= cnt
    print(f"Epoch {epoch+1}/{num_epochs}, Train_Loss: {t_loss:.4f}, Train_Acc: {t_acc:.4f}")

Epoch 1/15, Train_Loss: 0.5080, Train_Acc: 0.8217
Epoch 2/15, Train_Loss: 0.3372, Train_Acc: 0.8798
Epoch 3/15, Train_Loss: 0.2976, Train_Acc: 0.8921
Epoch 4/15, Train_Loss: 0.2633, Train_Acc: 0.9022
Epoch 5/15, Train_Loss: 0.2383, Train_Acc: 0.9122
Epoch 6/15, Train_Loss: 0.2198, Train_Acc: 0.9208
Epoch 7/15, Train_Loss: 0.1991, Train_Acc: 0.9276
Epoch 8/15, Train_Loss: 0.1864, Train_Acc: 0.9315
Epoch 9/15, Train_Loss: 0.1672, Train_Acc: 0.9382
Epoch 10/15, Train_Loss: 0.1543, Train_Acc: 0.9437
Epoch 11/15, Train_Loss: 0.1411, Train_Acc: 0.9489
Epoch 12/15, Train_Loss: 0.1316, Train_Acc: 0.9531
Epoch 13/15, Train_Loss: 0.1203, Train_Acc: 0.9570
Epoch 14/15, Train_Loss: 0.1132, Train_Acc: 0.9584
Epoch 15/15, Train_Loss: 0.1029, Train_Acc: 0.9633
